In [1]:
import pandas as pd
import random
from datetime import datetime, timedelta

# ==========================================
# 1. BACA DATASET DARI KAGGLE
# ==========================================
# Ganti 'healthcareTest.csv' dengan nama file CSV asli yang Anda download dari Kaggle
nama_file_csv = 'healthcareTest.csv' 

try:
    df_kaggle = pd.read_csv(nama_file_csv)
    # Di dataset Kaggle ini, ID pasien biasanya ada di kolom 'patIndex'
    # Jika namanya berbeda, silakan ganti 'patIndex' di bawah ini
    daftar_pasien = df_kaggle['patIndex'].unique().tolist()
    print(f"✅ Berhasil membaca file CSV. Ditemukan {len(daftar_pasien)} pasien unik.")
except FileNotFoundError:
    print(f"❌ File {nama_file_csv} tidak ditemukan! Pastikan file ada di folder yang sama.")
    exit()

# ==========================================
# 2. KONFIGURASI TINDAKAN MEDIS (DUMMY)
# ==========================================
# Kita buat rata-rata tiap pasien punya 1 sampai 3 tindakan medis
jumlah_record_dummy = len(daftar_pasien) * random.randint(1, 3) 

referensi_tindakan = [
    ("CPT-80053", "Laboratory", "Comprehensive Metabolic Panel", 50.00),
    ("CPT-71045", "Radiology", "Chest X-Ray, single view", 120.00),
    ("CPT-70450", "Radiology", "CT Scan, Head/Brain", 850.00),
    ("ICD10-0DTJ", "Surgery", "Appendectomy (Open)", 8500.00),
    ("CPT-93000", "Diagnostic", "Electrocardiogram (ECG)", 75.00)
]

def generate_tanggal_acak():
    start_date = datetime(2025, 1, 1)
    end_date = datetime(2026, 5, 24)
    selisih_hari = (end_date - start_date).days
    random_hari = random.randint(0, selisih_hari)
    return start_date + timedelta(days=random_hari)

# ==========================================
# 3. GENERATE DAN EXPORT KE .SQL
# ==========================================
nama_file_sql = "kaggle_medical_procedures_dummy.sql"

with open(nama_file_sql, 'w') as file:
    # Membuat DDL Tabel
    file.write("-- Membuat Tabel Medical_Procedures untuk melengkapi Dataset Kaggle\n")
    file.write("CREATE TABLE IF NOT EXISTS Medical_Procedures (\n")
    file.write("    Procedure_ID VARCHAR(20) PRIMARY KEY,\n")
    file.write("    patIndex VARCHAR(50), -- Relasi ke dataset Kaggle\n")
    file.write("    Procedure_Date DATE,\n")
    file.write("    Procedure_Code VARCHAR(20),\n")
    file.write("    Procedure_Category VARCHAR(50),\n")
    file.write("    Procedure_Description VARCHAR(255),\n")
    file.write("    Cost DECIMAL(10, 2)\n")
    file.write(");\n\n")
    
    file.write("-- Memasukkan Data Dummy (DML)\n")
    
    print(f"⏳ Sedang membuat {jumlah_record_dummy} record tindakan medis dummy...")
    
    for i in range(1, jumlah_record_dummy + 1):
        proc_id = f"TR-{1000 + i}"
        # Pilih pasien secara acak dari dataset Kaggle
        pat_id = random.choice(daftar_pasien) 
        tgl_tindakan = generate_tanggal_acak().strftime('%Y-%m-%d')
        
        tindakan = random.choice(referensi_tindakan)
        proc_code, proc_cat, proc_desc, harga_dasar = tindakan
        
        cost = round(harga_dasar * random.uniform(0.9, 1.1), 2)
        
        insert_query = (
            f"INSERT INTO Medical_Procedures "
            f"(Procedure_ID, patIndex, Procedure_Date, Procedure_Code, Procedure_Category, Procedure_Description, Cost) "
            f"VALUES ('{proc_id}', '{pat_id}', '{tgl_tindakan}', '{proc_code}', '{proc_cat}', '{proc_desc}', {cost});\n"
        )
        file.write(insert_query)

print(f"🎉 Selesai! File '{nama_file_sql}' berhasil dibuat dan siap di-import ke Database.")

✅ Berhasil membaca file CSV. Ditemukan 344 pasien unik.
⏳ Sedang membuat 1032 record tindakan medis dummy...
🎉 Selesai! File 'kaggle_medical_procedures_dummy.sql' berhasil dibuat dan siap di-import ke Database.
